# LoRA dialogue summarization

Fine-tune **SmolLM2-135M-Instruct** on DialogSum using LoRA.
In Colab, select a **GPU** and run the cells in order.
The README contains results from the completed experiment.


## 1. Setup

Install the libraries. If Colab requests a restart, restart and continue with the imports.


In [ ]:
%pip -q install transformers==4.57.1 peft==0.17.1 datasets==4.3.0 accelerate==1.11.0 trl==0.24.0 rouge-score==0.1.2


In [ ]:
import json
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

set_seed(42)
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
TRAIN_SIZE, EPOCHS = 1000, 3
TARGET_MODULES = ["q_proj", "v_proj"]
MAX_DIALOGUE_TOKENS, MAX_LENGTH, MAX_NEW_TOKENS = 320, 512, 128
GEN_BATCH_SIZE = 4
RUN_DIR = Path("lora_runs/qv_1000_examples")
if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise ValueError("Choose a new RUN_DIR so earlier experiment files are preserved.")
RUN_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 2. Prepare the data

Use 1,000 training examples and all 500 validation examples.
Keep the first 320 dialogue tokens; training uses at most 512 tokens per example.
Validation loss selects the checkpoint. The test split is used afterward.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
data = load_dataset("knkarthick/dialogsum")

def shorten_dialogue(dialogue):
    ids = tokenizer.encode(dialogue, add_special_tokens=False)
    return tokenizer.decode(ids[:MAX_DIALOGUE_TOKENS], clean_up_tokenization_spaces=False)

def make_prompt(dialogue):
    return [{"role": "user", "content": "Summarize this dialogue briefly:\n" + shorten_dialogue(dialogue)}]

def make_example(row):
    return {"prompt": make_prompt(row["dialogue"]),
            "completion": [{"role": "assistant", "content": row["summary"]}]}

train_raw = data["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
train_data = train_raw.map(make_example, remove_columns=train_raw.column_names)
val_data = data["validation"].map(make_example, remove_columns=data["validation"].column_names)
for split in (train_data, val_data):
    assert all(len(tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True)) < MAX_LENGTH
               for row in split), "Increase MAX_LENGTH: a prompt leaves no room for answer tokens."
print("Training:", len(train_data), "Validation:", len(val_data))


## 3. Add LoRA

Freeze the pretrained model and train small adapters in its query/value attention layers.
`r=8` sets the adapter rank; `lora_alpha=16` scales its contribution.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(device)
lora = LoraConfig(task_type="CAUSAL_LM", r=8, lora_alpha=16,
                  target_modules=TARGET_MODULES, lora_dropout=0.05)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 4. Train

Train for three epochs. Save each epoch and load the checkpoint with the lowest validation loss.
`completion_only_loss=True` scores the summary tokens.


In [ ]:
args = SFTConfig(
    output_dir=str(RUN_DIR / "checkpoints"), num_train_epochs=EPOCHS,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    learning_rate=2e-4, max_length=MAX_LENGTH, completion_only_loss=True,
    eos_token="<|im_end|>", packing=False,
    bf16=False, fp16=False, gradient_checkpointing=False,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=50, report_to="none", seed=42,
)
trainer = SFTTrainer(model=model, args=args, processing_class=tokenizer,
                     train_dataset=train_data, eval_dataset=val_data)
initial_val_loss = trainer.evaluate()["eval_loss"]
trainer.train()
model = trainer.model
epoch_results = pd.DataFrame([
    {"epoch": row["epoch"], "validation_loss": row["eval_loss"]}
    for row in trainer.state.log_history if "eval_loss" in row and row.get("epoch", 0) > 0
])
epoch_results.to_csv(RUN_DIR / "epoch_results.csv", index=False)
print("Initial validation loss:", initial_val_loss)
print(epoch_results.to_string(index=False))
print("Selected checkpoint:", trainer.state.best_model_checkpoint)
print("Selected validation loss:", trainer.state.best_metric)


## 5. Generate test summaries

The test split has 500 dialogues with three references each.
Compare the base model and selected adapter using the same prompts and greedy decoding.
Generate at most 128 new tokens. Left padding is used for generation.


In [ ]:
test_groups = {}
for row in data["test"]:
    test_groups.setdefault(row["dialogue"], []).append(row["summary"])
test_dialogues = list(test_groups)
test_references = list(test_groups.values())
test_prompts = [make_prompt(dialogue) for dialogue in test_dialogues]
test_shortened = sum(len(tokenizer.encode(text, add_special_tokens=False)) > MAX_DIALOGUE_TOKENS
                     for text in test_dialogues)
print("Test rows:", len(data["test"]), "Unique dialogues:", len(test_dialogues))
print("References per dialogue:", sorted(set(map(len, test_references))))
print("Shortened test inputs:", test_shortened)

def generate_summaries(prompts):
    model.eval()
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    predictions = []
    try:
        for start in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc="Generating"):
            texts = [tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
                     for p in prompts[start:start + GEN_BATCH_SIZE]]
            inputs = tokenizer(texts, padding=True, add_special_tokens=False, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
            predictions.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    finally:
        tokenizer.padding_side = old_padding_side
    return predictions

with model.disable_adapter():
    base_summaries = generate_summaries(test_prompts)
lora_summaries = generate_summaries(test_prompts)


## 6. Calculate ROUGE

Report ROUGE-1/2/L F1 on a 0-100 scale, with stemming.
Choose the best reference per metric, then average across dialogues.
ROUGE measures wording overlap; summaries can still contain factual errors.


In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def rouge_scores(predictions):
    assert len(predictions) == len(test_references) > 0
    per_dialogue = [scorer.score_multi(refs, prediction)
                    for refs, prediction in zip(test_references, predictions)]
    return {name: 100 * sum(scores[name].fmeasure for scores in per_dialogue) / len(per_dialogue)
            for name in ["rouge1", "rouge2", "rougeL"]}

scores = pd.DataFrame({"base": rouge_scores(base_summaries),
                       "selected_lora": rouge_scores(lora_summaries)}).T
scores.to_csv(RUN_DIR / "test_rouge.csv", index_label="model")
print(scores.round(2).to_string())
comparisons = pd.DataFrame({
    "dialogue": test_dialogues, "model_input": [shorten_dialogue(d) for d in test_dialogues],
    "references": [json.dumps(refs, ensure_ascii=False) for refs in test_references],
    "base_summary": base_summaries, "lora_summary": lora_summaries,
})
comparisons.to_csv(RUN_DIR / "test_summaries.csv", index=False)
display(comparisons[["base_summary", "lora_summary"]].head(3))


## 7. Save

Save the adapter, tokenizer, settings, scores, and generated summaries.
The adapter needs the original base model to run.


In [ ]:
model.save_pretrained(RUN_DIR / "selected_adapter")
tokenizer.save_pretrained(RUN_DIR / "selected_adapter")
(RUN_DIR / "training_args.json").write_text(trainer.args.to_json_string(), encoding="utf-8")
settings = {
    "model": MODEL_ID, "dataset": "knkarthick/dialogsum", "seed": 42,
    "training_examples": len(train_data), "validation_examples": len(val_data),
    "epochs": EPOCHS, "lora_targets": TARGET_MODULES,
    "max_dialogue_tokens": MAX_DIALOGUE_TOKENS, "max_new_tokens": MAX_NEW_TOKENS,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_loss": trainer.state.best_metric,
    "test_dialogues": len(test_dialogues), "shortened_test_inputs": test_shortened,
}
(RUN_DIR / "experiment.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print("Saved:", RUN_DIR.resolve())


### Download from Colab (optional)


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(str(RUN_DIR), "zip", RUN_DIR)
files.download(archive)


## Notes

This is a small learning project using one model and one seed.
69 of 500 test inputs were shortened in the recorded run.

[Model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) ·
[Dataset](https://huggingface.co/datasets/knkarthick/dialogsum) ·
[LoRA](https://huggingface.co/docs/peft/package_reference/lora)
